In [1]:
import os
SCRIPTS = "/content/drive/MyDrive/0_potato_project_v1/scripts"

if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

import importlib, sys
sys.path = [p for p in sys.path if p != SCRIPTS]
sys.path.insert(0, SCRIPTS)
importlib.invalidate_caches()

import torch
import config as C, data as D, model as M, train as T
for m in (C, D, M, T):
    importlib.reload(m)

D.setup_data()
df = D.load_manifest()

print("cuda    :", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "—")
print("images  :", len(df), "| per class:", df.y.value_counts().sort_index().tolist())

Mounted at /content/drive
restoring /content/drive/MyDrive/0_potato_project_v1/data/potato_raw/raw -> /content/potato
  Early_blight -> Potato___Early_blight
  Late_blight -> Potato___Late_blight
  Healthy -> Potato___healthy
cuda    : True | Tesla T4
images  : 2152 | per class: [1000, 1000, 152]


In [2]:
import time, numpy as np, torch.nn as nn
from torch.utils.data import DataLoader

EPOCHS_S1, EPOCHS_S2 = 8, 15          # medians from greyworld CV, rounded up
AUG = "greyworld"

C.AUG_MODE = AUG
T.set_seed(C.SEED)

# all 2152 images, train transforms, no split
tf_tr, _ = D.build_transforms(AUG)
full = D.PotatoDataset(df, np.arange(len(df)), transform=tf_tr)
loader = DataLoader(full, batch_size=C.BATCH_SIZE, shuffle=True, drop_last=True,
                    num_workers=C.NUM_WORKERS, pin_memory=True,
                    worker_init_fn=D._seed_worker)

w = D.class_weights(df.y.values).to(T.DEVICE)
net = M.set_stage(M.build_model(), stage=1).to(T.DEVICE)
crit = nn.CrossEntropyLoss(weight=w)
scaler = torch.amp.GradScaler(enabled=T.AMP)

print(f"training on {len(full)} images | weights {[round(v,3) for v in w.tolist()]}")
t0 = time.time()

for stage, n_ep in [(1, EPOCHS_S1), (2, EPOCHS_S2)]:
    M.set_stage(net, stage)
    opt = torch.optim.AdamW(M.param_groups(net, stage), weight_decay=1e-4)
    for ep in range(1, n_ep + 1):
        te = time.time()
        loss = T.train_one_epoch(net, loader, crit, opt, scaler)
        print(f"  s{stage} e{ep:>2}  train {loss:.4f}  {time.time()-te:.0f}s")

print(f"\ntotal: {(time.time()-t0)/60:.1f} min")

Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-8738ca79.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-8738ca79.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 69.9MB/s]


training on 2152 images | weights [0.717, 0.717, 4.719]
  s1 e 1  train 0.1620  23s
  s1 e 2  train 0.0593  17s
  s1 e 3  train 0.0434  17s
  s1 e 4  train 0.0358  18s
  s1 e 5  train 0.0634  18s
  s1 e 6  train 0.0340  17s
  s1 e 7  train 0.0195  17s
  s1 e 8  train 0.0199  17s
  s2 e 1  train 0.0811  19s
  s2 e 2  train 0.0639  19s
  s2 e 3  train 0.0465  18s
  s2 e 4  train 0.0354  17s
  s2 e 5  train 0.0270  18s
  s2 e 6  train 0.0235  18s
  s2 e 7  train 0.0160  19s
  s2 e 8  train 0.0262  18s
  s2 e 9  train 0.0187  18s
  s2 e10  train 0.0262  17s
  s2 e11  train 0.0135  17s
  s2 e12  train 0.0161  19s
  s2 e13  train 0.0156  18s
  s2 e14  train 0.0147  17s
  s2 e15  train 0.0134  17s

total: 6.9 min


In [3]:
from pathlib import Path
import json

FINAL_DIR = C.RESULTS / "final"
FINAL_DIR.mkdir(parents=True, exist_ok=True)

path = M.save_checkpoint(
    FINAL_DIR / "potato_mobilenetv3_final.pt", net,
    fold=-1, stage=2, epoch=EPOCHS_S2, aug_mode=AUG,
    metrics={"source": "5-fold CV, greyworld run",
             "cv_macro_f1": 0.9920, "cv_macro_f1_std": 0.0072,
             "cv_healthy_recall": 0.9935, "cv_healthy_recall_std": 0.0144,
             "confidence_threshold": 0.90})

# provenance alongside the weights, readable without torch
json.dump({
    "arch": "mobilenet_v3_large",
    "trained_on": int(len(df)),
    "class_to_idx": C.CLASS_TO_IDX,
    "img_size": C.IMG_SIZE,
    "aug_mode": AUG,
    "epochs": {"stage1": EPOCHS_S1, "stage2": EPOCHS_S2},
    "normalisation": {"mean": C.IMAGENET_MEAN, "std": C.IMAGENET_STD},
    "cv_macro_f1": "0.9920 ± 0.0072",
    "cv_healthy_recall": "0.9935 ± 0.0144",
    "confidence_threshold": 0.90,
    "note": "No held-out set: trained on all data. Metrics are from 5-fold CV "
            "under the same configuration. Grey-world normalisation must be "
            "applied at inference.",
}, open(FINAL_DIR / "model_card.json", "w"), indent=2)

# reload check — proves the file serves correctly from disk
net2, meta = M.load_checkpoint(path, device=T.DEVICE)
x = torch.randn(1, 3, C.IMG_SIZE, C.IMG_SIZE).to(T.DEVICE)
with torch.no_grad():
    p = torch.softmax(net2(x), 1)[0]

print("saved     :", path.relative_to(C.PROJECT), f"{path.stat().st_size/1024**2:.1f} MB")
print("aug_mode  :", meta["aug_mode"], "| classes:", meta["class_to_idx"])
print("reload ok :", not net2.training, "| output sums to 1:",
      abs(float(p.sum()) - 1) < 1e-5)
print("files     :", sorted(f.name for f in FINAL_DIR.iterdir()))

saved     : results/final/potato_mobilenetv3_final.pt 16.2 MB
aug_mode  : greyworld | classes: {'Potato___Early_blight': 0, 'Potato___Late_blight': 1, 'Potato___healthy': 2}
reload ok : True | output sums to 1: True
files     : ['model_card.json', 'potato_mobilenetv3_final.pt']
